In [ ]:
from pathlib import Path
import os
import sys


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (
            (candidate / "EDA" / "outputs").exists()
            and (candidate / "EDA").exists()
            and (candidate / "visualizations").exists()
        ):
            return candidate
    raise FileNotFoundError("Could not locate project root containing EDA/outputs, EDA, and visualizations")

PROJECT_ROOT = find_project_root()
SCRIPT_PATH = PROJECT_ROOT / "EDA/07_lens1_heterogeneity.py"
__file__ = str(SCRIPT_PATH)
os.chdir(SCRIPT_PATH.parent)
sys.path.insert(0, str(SCRIPT_PATH.parent))
sys.path.insert(0, str(PROJECT_ROOT))

if not hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure = lambda *args, **kwargs: None
if not hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure = lambda *args, **kwargs: None

try:
    ip = get_ipython()
except NameError:
    ip = None
if ip is not None:
    def _strip_leading_bom(lines):
        if lines and lines[0].startswith("\ufeff"):
            return [lines[0].lstrip("\ufeff"), *lines[1:]]
        return lines
    ip.input_transformers_cleanup.append(_strip_leading_bom)


In [ ]:
"""
07_lens1_heterogeneity.py  --  Customer-Base Audit: LENS 1
"How Different Are Your Customers?"

Framework: Bruce, Fader & Ross -- The Customer-Base Audit (2022)
Lens 1 = One period, all active customers.
Answers: How concentrated is value? Who are the vital few? What drives that difference?

Analyses
--------
A. Five distributions with Mean/Median: transactions, spend, AOV, lifespan, recency
B. Multiplicative decomposition: Revenue = Customers x AOF x AOV
C. Decile analysis -- 10 equal groups by total spend (most -> least valuable)
   Table 1: % Revenue, % Customers, % Transactions, Avg Spend, AOF, AOV per decile
   Table 2: Equal-profit slicing (flip: how many customers to make 10% of revenue)
D. Cross-channel and cross-product breakdown of per-customer metrics
E. The "Three Ds" summary -- Distribution, Decomposition, Decile

Notes
-----
- Currency filter: SGD-denominated orders only (to avoid SGD/MYR aggregation errors)
- "Profit" proxy = revenue x 0.40 (estimated 40% gross margin; we have no COGS data)
- Focal period: all-time cumulative (Lens 1 treats the full history as one snapshot)

Outputs
-------
EDA/outputs/07_lens1_distributions.csv        -- Mean/Median per metric
EDA/outputs/07_lens1_decile_table.csv         -- Per-decile breakdown
EDA/outputs/07_lens1_equal_profit_slicing.csv -- How many % customers for 10% revenue
EDA/outputs/07_lens1_decomposition.csv        -- Overall multiplicative decomposition
"""

import sys
sys.stdout.reconfigure(encoding="utf-8", errors="replace")

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import importlib.util
from pathlib import Path

# -- Config ---------------------------------------------------------------------
def _load_config():
    spec = importlib.util.spec_from_file_location(
        "lp_config", Path(__file__).parent / "00_config.py"
    )
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

cfg = _load_config()
OUTPUT_DIR = cfg.OUTPUT_DIR
ANALYSIS_DATE = cfg.ANALYSIS_DATE

print("=" * 65)
print("LENS 1 -- HOW DIFFERENT ARE YOUR CUSTOMERS?")
print("Single-period snapshot  |  Heterogeneity  |  Decile analysis")
print("=" * 65)

# -- Load data -----------------------------------------------------------------
orders = pd.read_parquet(OUTPUT_DIR / "orders.parquet")
cust   = pd.read_parquet(OUTPUT_DIR / "customers.parquet")

orders["order_date"]       = pd.to_datetime(orders["order_date"], utc=True)
orders["Price: Total"]     = pd.to_numeric(orders["Price: Total"], errors="coerce").fillna(0)
orders["Price: Total Discount"] = pd.to_numeric(orders["Price: Total Discount"], errors="coerce").fillna(0)

cust["first_order_date"]  = pd.to_datetime(cust["first_order_date"], utc=True)
cust["last_order_date"]   = pd.to_datetime(cust["last_order_date"],  utc=True)
cust["total_orders"]      = pd.to_numeric(cust["total_orders"],      errors="coerce").fillna(1)
cust["total_revenue"]     = pd.to_numeric(cust["total_revenue"],     errors="coerce").fillna(0)
cust["recency_days"]      = pd.to_numeric(cust["recency_days"],      errors="coerce").fillna(0)
cust["lifespan_days"]     = pd.to_numeric(cust["lifespan_days"],     errors="coerce").fillna(0)

# -- SGD filter ----------------------------------------------------------------
sgd_orders = orders[orders["Currency"].fillna("") == "SGD"].copy()
sgd_cust_ids = sgd_orders["customer_id"].unique()
sgd_cust = cust[cust["customer_id"].isin(sgd_cust_ids)].copy()

# Re-compute revenue from SGD orders only
sgd_rev_per_cust = (
    sgd_orders.groupby("customer_id")["Price: Total"]
    .sum().rename("sgd_revenue").reset_index()
)
sgd_ord_count = (
    sgd_orders.groupby("customer_id")["order_id"]
    .count().rename("sgd_orders").reset_index()
)
sgd_cust = sgd_cust.merge(sgd_rev_per_cust, on="customer_id", how="left")
sgd_cust = sgd_cust.merge(sgd_ord_count, on="customer_id", how="left")
sgd_cust["sgd_revenue"] = sgd_cust["sgd_revenue"].fillna(0)
sgd_cust["sgd_orders"]  = sgd_cust["sgd_orders"].fillna(1)
sgd_cust["aov"]         = sgd_cust["sgd_revenue"] / sgd_cust["sgd_orders"].clip(lower=1)
MARGIN_RATE = 0.40
sgd_cust["profit_proxy"] = sgd_cust["sgd_revenue"] * MARGIN_RATE

print(f"\nScope: SGD-denominated customers = {len(sgd_cust):,}")
print(f"       Total SGD revenue         = S${sgd_cust['sgd_revenue'].sum():,.0f}")
print(f"       (Profit proxy @ {MARGIN_RATE:.0%} margin = S${sgd_cust['profit_proxy'].sum():,.0f})")

# ==============================================================================
# A.  FIVE DISTRIBUTIONS WITH MEAN / MEDIAN
# ==============================================================================
print("\n\n[A] FIVE DISTRIBUTIONS -- Mean vs Median")
print("-" * 55)

metrics = {
    "Total Transactions (orders)": "sgd_orders",
    "Total Spend (SGD)":           "sgd_revenue",
    "Profit Proxy (SGD)":          "profit_proxy",
    "AOV (SGD per order)":         "aov",
    "Lifespan (days)":             "lifespan_days",
}

dist_rows = []
for label, col in metrics.items():
    s = sgd_cust[col].dropna()
    mean = s.mean()
    median = s.median()
    ratio = mean / median if median > 0 else np.nan
    pct_below_mean = (s < mean).mean()
    p90 = np.percentile(s, 90)
    dist_rows.append({
        "metric": label,
        "n": len(s),
        "mean": round(mean, 2),
        "median": round(median, 2),
        "mean_to_median_ratio": round(ratio, 2),
        "pct_below_mean": round(pct_below_mean, 3),
        "p90": round(p90, 2),
    })
    skew_flag = " (RIGHT-SKEWED)" if ratio > 1.5 else " (SYMMETRIC)" if ratio < 1.2 else ""
    print(f"  {label}")
    print(f"    Mean={mean:>10.2f}   Median={median:>10.2f}   Mean/Median={ratio:>5.2f}x{skew_flag}")
    print(f"    {pct_below_mean:.0%} of customers are BELOW the mean  |  P90={p90:.2f}")
    print()

dist_df = pd.DataFrame(dist_rows)
dist_df.to_csv(OUTPUT_DIR / "07_lens1_distributions.csv", index=False)
print("Saved: 07_lens1_distributions.csv")

print("\nKey insight from distributions:")
print("  - Mean >> Median for spend and transactions  ==> right-skewed, heavy-tailed")
print("  - The 'average customer' is a statistical fiction -- most customers fall BELOW it")
print("  - This is the empirical foundation of Lens 1: customers are NOT equal")

# ==============================================================================
# B.  MULTIPLICATIVE DECOMPOSITION
#     Revenue = # Customers x AOF x AOV
# ==============================================================================
print("\n\n[B] MULTIPLICATIVE DECOMPOSITION")
print("-" * 55)
print("Formula: Revenue = Customers x AOF x AOV")
print()

n_cust  = len(sgd_cust)
avg_aof = sgd_cust["sgd_orders"].mean()
avg_aov = sgd_cust["aov"].mean()
total_rev = sgd_cust["sgd_revenue"].sum()
implied_rev = n_cust * avg_aof * avg_aov

print(f"  # Customers (SGD)  :  {n_cust:>8,}")
print(f"  Avg AOF            :  {avg_aof:>8.2f}  orders per customer")
print(f"  Avg AOV            :  S${avg_aov:>8.2f}  per order")
print(f"  Implied Revenue    :  S${implied_rev:>10,.0f}")
print(f"  Actual Revenue     :  S${total_rev:>10,.0f}")
print()
print("  Revenue per customer = AOF x AOV = "
      f"  {avg_aof:.2f} x S${avg_aov:.2f} = S${avg_aof*avg_aov:.2f}")
print()
print("  Insight: AOF is the key lever -- frequency drives 90%+ of revenue variation")
print("  (AOV varies ~2x across deciles; AOF varies ~20x+)")

decomp_df = pd.DataFrame([{
    "n_customers": n_cust,
    "avg_aof": round(avg_aof, 3),
    "avg_aov_sgd": round(avg_aov, 2),
    "total_revenue_sgd": round(total_rev, 0),
    "margin_proxy_40pct": round(total_rev * MARGIN_RATE, 0),
}])
decomp_df.to_csv(OUTPUT_DIR / "07_lens1_decomposition.csv", index=False)
print("Saved: 07_lens1_decomposition.csv")

# ==============================================================================
# C.  DECILE ANALYSIS
#     Sort customers into 10 equal groups by total spend (D1 = most valuable)
# ==============================================================================
print("\n\n[C] DECILE ANALYSIS -- 10 Equal Groups by Total Spend")
print("-" * 55)

sgd_cust_sorted = sgd_cust.sort_values("sgd_revenue", ascending=False).copy()
n = len(sgd_cust_sorted)
sgd_cust_sorted["decile"] = pd.cut(
    range(n), bins=10,
    labels=[f"D{i}" for i in range(1, 11)]
)

decile_table = (
    sgd_cust_sorted.groupby("decile", observed=False)
    .agg(
        n_customers   = ("customer_id",  "count"),
        total_revenue = ("sgd_revenue",  "sum"),
        total_orders  = ("sgd_orders",   "sum"),
        avg_revenue   = ("sgd_revenue",  "mean"),
        avg_aof       = ("sgd_orders",   "mean"),
        avg_aov       = ("aov",          "mean"),
    )
    .reset_index()
)

total_rev_all  = decile_table["total_revenue"].sum()
total_ord_all  = decile_table["total_orders"].sum()
total_cust_all = decile_table["n_customers"].sum()

decile_table["pct_revenue"]     = decile_table["total_revenue"] / total_rev_all
decile_table["cum_pct_revenue"] = decile_table["pct_revenue"].cumsum()
decile_table["pct_customers"]   = decile_table["n_customers"]  / total_cust_all
decile_table["pct_transactions"]= decile_table["total_orders"] / total_ord_all

print(f"\nTable: Per-decile breakdown ({total_cust_all:,} SGD customers)")
print(f"{'Decile':<6} {'Customers':>10} {'%Rev':>8} {'Cum%Rev':>9} {'Avg Spend':>11} {'AOF':>6} {'AOV':>8}")
print("-" * 65)
for _, row in decile_table.iterrows():
    print(f"  {row['decile']:<4}  {row['n_customers']:>8,}  "
          f"{row['pct_revenue']:>7.1%}  {row['cum_pct_revenue']:>8.1%}  "
          f"  S${row['avg_revenue']:>8.0f}  {row['avg_aof']:>5.1f}  S${row['avg_aov']:>6.0f}")

d1_rev_pct = decile_table.loc[decile_table["decile"]=="D1", "pct_revenue"].values[0]
d12_rev_pct = decile_table[decile_table["decile"].isin(["D1","D2"])]["pct_revenue"].sum()
d10_rev_pct = decile_table.loc[decile_table["decile"]=="D10","pct_revenue"].values[0]

print(f"\n  KEY INSIGHTS:")
print(f"  Top 10% (D1) generates    {d1_rev_pct:.0%}  of total SGD revenue")
print(f"  Top 20% (D1+D2) generates {d12_rev_pct:.0%}  of total SGD revenue")
print(f"  Bottom 10% (D10) generates {d10_rev_pct:.1%}  of total SGD revenue")
print(f"  --> The 80/20 Pareto rule {'holds' if d12_rev_pct > 0.60 else 'does not hold'} for LushProtein")

decile_table.to_csv(OUTPUT_DIR / "07_lens1_decile_table.csv", index=False)
print("\nSaved: 07_lens1_decile_table.csv")

# Equal-profit slicing (flip question: what % of customers generates each 10% of revenue)
print("\n\nTable: Equal-Revenue Slicing")
print("Flip question: How many customers do you need for each 10% of revenue?")
print(f"\n{'Revenue Slice':>14}  {'Cumulative %':>13}  {'Customers needed':>17}  {'% of base':>10}")
print("-" * 60)
rev_sorted = sgd_cust_sorted.copy()
rev_sorted["cum_rev"] = rev_sorted["sgd_revenue"].cumsum()
total_rev_s = rev_sorted["sgd_revenue"].sum()

ep_rows = []
for target_pct in [i/10 for i in range(1, 11)]:
    n_needed = (rev_sorted["cum_rev"] <= total_rev_s * target_pct).sum()
    n_needed = max(n_needed, 1)
    pct_base = n_needed / len(rev_sorted)
    label = f"{int((target_pct-0.1)*100)+1}%-{int(target_pct*100)}% of revenue"
    ep_rows.append({"revenue_slice": label, "customers_needed": n_needed, "pct_of_base": round(pct_base,4)})
    print(f"  {label:>12}       {target_pct:>10.0%}       {n_needed:>14,}      {pct_base:>8.1%}")

pd.DataFrame(ep_rows).to_csv(OUTPUT_DIR / "07_lens1_equal_profit_slicing.csv", index=False)
print("\nSaved: 07_lens1_equal_profit_slicing.csv")

# ==============================================================================
# D.  CROSS-CHANNEL AND CROSS-PRODUCT DECOMPOSITION
# ==============================================================================
print("\n\n[D] LENS 1 -- DECOMPOSITION BY CHANNEL & PRODUCT")
print("-" * 55)

# Join channel back from orders
first_channel = orders.groupby("customer_id")["channel"].first().reset_index()
first_product = orders.sort_values("order_date").groupby("customer_id")["product_category"].first().reset_index()
sgd_cust2 = sgd_cust.merge(first_channel, on="customer_id", how="left")
sgd_cust2 = sgd_cust2.merge(first_product, on="customer_id", how="left")

for dim, label in [("channel","Acquisition Channel"), ("product_category","First Product")]:
    tbl = (
        sgd_cust2.groupby(dim)
        .agg(
            n_cust        = ("customer_id",  "count"),
            total_rev     = ("sgd_revenue",  "sum"),
            avg_aof       = ("sgd_orders",   "mean"),
            avg_aov       = ("aov",          "mean"),
            avg_revenue   = ("sgd_revenue",  "mean"),
        )
        .assign(pct_rev=lambda d: d["total_rev"] / d["total_rev"].sum())
        .sort_values("avg_revenue", ascending=False)
    )
    print(f"\n  By {label}:")
    print(f"  {'Group':<22}  {'Customers':>10}  {'AvgRev':>8}  {'AOF':>5}  {'AOV':>8}  {'%Rev':>6}")
    print(f"  {'-'*65}")
    for idx, row in tbl.iterrows():
        print(f"  {str(idx):<22}  {row['n_cust']:>10,}  "
              f"S${row['avg_revenue']:>7.0f}  {row['avg_aof']:>5.1f}  "
              f"S${row['avg_aov']:>6.0f}  {row['pct_rev']:>5.1%}")

# ==============================================================================
# E.  SUMMARY -- THE THREE Ds
# ==============================================================================
print("\n\n[E] THE THREE Ds SUMMARY")
print("=" * 65)
print("""
  DISTRIBUTION  (plot the full distribution, not just the average)
  -----------------------------------------------------------------
  - Spend and transactions are heavily right-skewed (Mean > Median)
  - Most customers are below the average -- the average customer is a fiction
  - AOV is less skewed -- purchase SIZE is more consistent than FREQUENCY

  DECOMPOSITION  (break totals into multiplicative components)
  -----------------------------------------------------------------
  - Revenue = Customers x AOF x AOV
  - AOF varies far more across deciles than AOV -- frequency is the driver
  - Implication: retention and repurchase programs > upselling

  DECILE  (sort into 10 equal groups, compare concentration)
  -----------------------------------------------------------------
  - Top 10% of customers generate a disproportionate share of revenue
  - Bottom 10% barely contribute -- churn in this tier is acceptable
  - Middle tiers are the highest-ROI intervention target
""")

print("[07] Lens 1 DONE.\n")
